In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

from src.models import (
    BarotropicPowerDiffusionParams,
    BarotropicPowerDiffusionModel,
)

In [2]:
params = BarotropicPowerDiffusionParams(
    omega=-1.2,
    alpha=0.3,
    beta=1.5,
    rho0=1.0,
    a0=1.0,
    t0=0.0,
)

model = BarotropicPowerDiffusionModel(
    params
)

background = model.solve_background_t(
    t_end=2.0,
    n_points=1000,
    rho_max=1e10,
    a_max=1e20,
    max_step=0.01,
)

background.head()

,t,rho,a,H,Hdot
0,0.000000,1.000000,1.000000,0.658281,0.100000
1,0.002002,1.000546,1.001319,0.658481,0.100055
2,0.004004,1.001091,1.002640,0.658681,0.100109
3,0.006006,1.001638,1.003963,0.658882,0.100164
4,0.008008,1.002185,1.005289,0.659082,0.100218


In [3]:
print(background.attrs)
print(background.tail())

{'solver_success': True, 'solver_message': 'The solver successfully reached the end of the integration interval.', 'termination_event': None, 'termination_time': 2.0, 'requested_t_end': 2.0, 'reached_t_end': True, 'method': 'DOP853', 'rtol': 1e-09, 'atol': 1e-11, 'rho_min': 1e-12, 'rho_max': 10000000000.0, 'a_max': 1e+20, 'boundary_tol': 1e-08}
            t       rho         a         H      Hdot
995  1.991992  1.842103  4.730552  0.929544  0.184210
996  1.993994  1.843381  4.739365  0.929913  0.184338
997  1.995996  1.844660  4.748198  0.930282  0.184466
998  1.997998  1.845940  4.757051  0.930652  0.184594
999  2.000000  1.847221  4.765925  0.931021  0.184722


In [4]:
turnaround_params = (
    BarotropicPowerDiffusionParams(
        omega=0.0,
        alpha=-0.5,
        beta=0.5,
        rho0=1.0,
        a0=1.0,
        t0=0.0,
    )
)

turnaround_model = (
    BarotropicPowerDiffusionModel(
        turnaround_params
    )
)

signed_background = (
    turnaround_model
    .solve_signed_background_t(
        t_end=20.0,
        n_points=2000,
        rho_min=1e-12,
        rho_max=1e12,
        a_min=1e-12,
        a_max=1e12,
        max_step=0.005,
    )
)

In [5]:
print(
    "Éxito:",
    signed_background.attrs[
        "solver_success"
    ],
)

print(
    "Cruces H=0:",
    signed_background.attrs[
        "H_zero_times"
    ],
)

print(
    "Residuo máximo:",
    signed_background.attrs[
        "maximum_constraint_residual"
    ],
)

signed_background.tail()

Éxito: True
Cruces H=0: [1.9051586888313599]
Residuo máximo: 8.07251676375079e-08


,t,rho,a,H,Hdot,constraint_residual
1995,4.974385,1.340065e+04,0.049665,-66.690251,-6.700325e+03,-3.121203e-11
1996,4.976879,2.382152e+04,0.041013,-88.965069,-1.191076e+04,1.652411e-11
1997,4.979372,5.359013e+04,0.031312,-133.509523,-2.679506e+04,1.409375e-11
1998,4.981866,2.142612e+05,0.019735,-267.101596,-1.071306e+05,-3.031538e-11
1999,4.984359,1.000000e+12,0.000118,-577350.125196,-5.000000e+11,-8.399235e-12


In [6]:
model_checks = {
    "expanding_solver_success": bool(
        background.attrs[
            "solver_success"
        ]
    ),
    "signed_solver_success": bool(
        signed_background.attrs[
            "solver_success"
        ]
    ),
    "H_zero_detected": (
        len(
            signed_background.attrs[
                "H_zero_times"
            ]
        )
        >= 1
    ),
    "friedmann_constraint": (
        signed_background.attrs[
            "maximum_constraint_residual"
        ]
        < 1e-6
    ),
}

model_checks

{'expanding_solver_success': True,
 'signed_solver_success': True,
 'H_zero_detected': True,
 'friedmann_constraint': True}

In [3]:
from src.singularities import (
    ClassificationResult,
    ClassificationStatus,
    KnownCosmologicalBehavior,
    SingularityType,
    describe_classification,
)

In [4]:
big_rip_result = ClassificationResult(
    standard_type=SingularityType.TYPE_I,
    behavior_label=(
        KnownCosmologicalBehavior
        .TYPE_I_BIG_RIP
    ),
    status=ClassificationStatus.ROBUST,
    is_singularity=True,
    finite_time=True,
    confidence=0.99,
    observed_behavior={
        "rho": "diverges",
        "a": "diverges",
        "H": "diverges",
        "Hdot": "diverges",
    },
    evidence={
        "threshold_times_converge": True,
        "friedmann_constraint_satisfied": True,
    },
    notes=(
        "La divergencia ocurre en un tiempo futuro finito."
    ),
)

print(
    describe_classification(
        big_rip_result
    )
)

type_i_big_rip: Type I / Big Rip; singularidad; tiempo finito; estado=robust; confianza=0.990.


In [5]:
new_behavior_result = ClassificationResult(
    standard_type=None,
    behavior_label=(
        "new_diffusion_dominated_branch"
    ),
    status=(
        ClassificationStatus
        .NEW_BEHAVIOR_CANDIDATE
    ),
    is_singularity=None,
    finite_time=False,
    confidence=0.70,
    observed_behavior={
        "rho": "tends_to_zero",
        "a": "diverges",
        "H": "diverges",
        "Hdot": "tends_to_zero",
    },
    evidence={
        "stable_under_threshold_changes": True,
        "time_limit": "infinite",
    },
    notes=(
        "No coincide directamente con los tipos I--IV."
    ),
)

print(
    describe_classification(
        new_behavior_result
    )
)

new_diffusion_dominated_branch: sin clasificación estándar; naturaleza singular no determinada; tiempo infinito; estado=new_behavior_candidate; confianza=0.700.


In [6]:
singularities_checks = {
    "standard_result_created": (
        big_rip_result.standard_type
        is SingularityType.TYPE_I
    ),
    "new_label_accepted": (
        new_behavior_result.behavior_label
        == "new_diffusion_dominated_branch"
    ),
    "new_behavior_without_standard_type": (
        new_behavior_result.standard_type
        is None
    ),
    "observed_behavior_is_preserved": (
        new_behavior_result
        .observed_behavior["H"]
        == "diverges"
    ),
}

singularities_checks

{'standard_result_created': True,
 'new_label_accepted': True,
 'new_behavior_without_standard_type': True,
 'observed_behavior_is_preserved': True}

In [4]:
from src.models import (
    BarotropicPowerDiffusionParams,
    BarotropicPowerDiffusionModel,
)

from src.classifiers import (
    classify_barotropic_power_diffusion,
    classify_phantom_q0_by_nu,
    classify_phantom_qalpha_rho_by_nu,
)

from src.singularities import (
    SingularityType,
)

In [10]:
old_classifier_checks = {
    "q0_big_rip": (
        classify_phantom_q0_by_nu(
            nu=0.75,
            A=1.0,
        )
        is SingularityType.TYPE_I
    ),
    "q0_type_III": (
        classify_phantom_q0_by_nu(
            nu=1.5,
            A=1.0,
        )
        is SingularityType.TYPE_III
    ),
    "qalpha_type_IV": (
        classify_phantom_qalpha_rho_by_nu(
            nu=0.25,
            A=1.0,
            alpha=0.3,
        )
        is SingularityType.TYPE_IV
    ),
}

old_classifier_checks

{'q0_big_rip': True, 'q0_type_III': True, 'qalpha_type_IV': True}

In [4]:
big_rip_params = (
    BarotropicPowerDiffusionParams(
        omega=-1.2,
        alpha=0.3,
        beta=1.5,
        rho0=1.0,
        a0=1.0,
        t0=0.0,
    )
)

big_rip_model = (
    BarotropicPowerDiffusionModel(
        big_rip_params
    )
)

big_rip_background = (
    big_rip_model.solve_background_t(
        t_end=20.0,
        n_points=2000,
        rho_max=1e10,
        a_max=1e50,
        max_step=0.01,
    )
)

In [5]:
big_rip_evidence = {
    "time_limit": "finite",
    "rho_limit": "diverges",
    "a_limit": "diverges",
    "H_limit": "diverges",
    "Hdot_limit": "diverges",
    "stable_under_threshold_changes": True,
    "confidence": 0.95,
}

In [6]:
big_rip_classification = (
    classify_barotropic_power_diffusion(
        background=big_rip_background,
        params=big_rip_params,
        limit_evidence=(
            big_rip_evidence
        ),
    )
)

big_rip_classification.as_dict()

{'standard_type': 'Type I / Big Rip',
 'behavior_label': 'type_i_big_rip',
 'status': 'robust',
 'is_singularity': True,
 'finite_time': True,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'finite',
  'rho': 'diverges',
  'a': 'diverges',
  'H': 'diverges',
  'Hdot': 'diverges',
  'rho_dot': 'unknown',
  'Hddot': 'unknown',
  'termination_event': 'a_max',
  'solver_success': True,
  'reached_t_end': False,
  'H_zero_crossings': 0,
  'expansion_to_contraction': False,
  'initial_H_sign': 1,
  'final_H_sign': 1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 0.6582805886043833,
   'Hdot': 0.09999999999999998},
  'final_values': {'t': 10.15343270037758,
   'rho': 4673.674758998313,
   'a': 9.99999999999906e+49,
   'H': 183.05499856636737,
   'Hdot': 467.3674758998312},
  'tail_relative_spread': {'rho': 0.9174660784442025,
   'a': 1.0,
   'H': 0.8373786326827739,
   'Hdot': 0.9174660784442025}},
 'evidence': {'time_limit': 'finite',
  'rho_limit': 'diverges'

In [7]:
new_behavior_evidence = {
    "time_limit": "infinite",
    "rho_limit": "zero",
    "a_limit": "diverges",
    "H_limit": "diverges",
    "Hdot_limit": "zero",
    "stable_under_threshold_changes": True,
    "confidence": 0.90,
}

In [12]:
params_F2 = BarotropicPowerDiffusionParams(
    omega=-1.2,
    alpha=4.0,
    beta=-1.0,
    rho0=1.0,
    a0=1.0,
    t0=0.0,
)

model_F2 = BarotropicPowerDiffusionModel(
    params_F2
)

background_F2 = model_F2.solve_background_t(
    t_end=20.0,
    n_points=2000,
    rho_min=1e-12,
    rho_max=1e12,
    a_max=1e100,
    boundary_tol=1e-8,
    max_step=0.02,
)

In [13]:
new_behavior_classification = (
    classify_barotropic_power_diffusion(
        background=background_F2,
        params=params_F2,
        limit_evidence=(
            new_behavior_evidence
        ),
    )
)

new_behavior_classification.as_dict()

{'standard_type': None,
 'behavior_label': 'new_behavior_candidate',
 'status': 'new_behavior_candidate',
 'is_singularity': None,
 'finite_time': None,
 'confidence': 0.9,
 'observed_behavior': {'time_limit': 'diverges',
  'rho': 'zero',
  'a': 'diverges',
  'H': 'diverges',
  'Hdot': 'zero',
  'rho_dot': 'unknown',
  'Hddot': 'unknown',
  'termination_event': None,
  'solver_success': True,
  'reached_t_end': True,
  'H_zero_crossings': 0,
  'expansion_to_contraction': False,
  'initial_H_sign': 1,
  'final_H_sign': 1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 1.2909944487358056,
   'Hdot': 0.09999999999999998},
  'final_values': {'t': 20.0,
   'rho': 0.281486747565625,
   'a': 8037918025434863.0,
   'H': 2.1978585950352323,
   'Hdot': 0.028148674756562492},
  'tail_relative_spread': {'rho': 0.05388243440040045,
   'a': 0.9866883165243765,
   'H': 0.026208241527267494,
   'Hdot': 0.05388243440040046}},
 'evidence': {'time_limit': 'infinite',
  'rho_limit': 'z

In [14]:
print(type(params_F2))
print(type(background_F2))

print(
    "Éxito del integrador:",
    background_F2.attrs["solver_success"],
)

print(
    "Evento final:",
    background_F2.attrs["termination_event"],
)

background_F2.tail()

<class 'src.models.BarotropicPowerDiffusionParams'>
<class 'pandas.core.frame.DataFrame'>
Éxito del integrador: True
Evento final: None


,t,rho,a,H,Hdot
1995,19.959980,0.281787,7.361284e+15,2.196731,0.028179
1996,19.969985,0.281712,7.524874e+15,2.197013,0.028171
1997,19.979990,0.281637,7.692121e+15,2.197295,0.028164
1998,19.989995,0.281562,7.863108e+15,2.197577,0.028156
1999,20.000000,0.281487,8.037918e+15,2.197859,0.028149


In [5]:
from src.barotropic_analysis import (
    compute_critical_densities,
    initial_density_direction,
    scan_density_targets,
    state_at_density,
)

In [6]:
critical_F2 = compute_critical_densities(
    params_F2
)

critical_F2.as_dict()

NameError: name 'params_F2' is not defined

In [7]:
from src.models import (
    BarotropicPowerDiffusionParams,
    BarotropicPowerDiffusionModel,
)

from src.barotropic_analysis import (
    compute_critical_densities,
    initial_density_direction,
    scan_density_targets,
    state_at_density,
)

In [8]:
params_F2 = BarotropicPowerDiffusionParams(
    omega=-1.2,
    alpha=4.0,
    beta=-1.0,
    rho0=1.0,
    a0=1.0,
    t0=0.0,
)

In [9]:
critical_F2 = compute_critical_densities(
    params_F2
)

critical_F2.as_dict()

{'rho_F': None,
 'rho_D': 2.0,
 'F_degenerate': False,
 'D_degenerate': False,
 'notes': ''}

In [10]:
initial_density_direction(
    params_F2
)

-1

In [11]:
F2_density_targets = [
    1e-2,
    1e-4,
    1e-6,
    1e-8,
    1e-10,
    1e-12,
]

F2_density_scan = scan_density_targets(
    params=params_F2,
    rho_targets=F2_density_targets,
    require_future=True,
    strict=False,
)

In [12]:
F2_density_scan[
    [
        "success",
        "rho",
        "t",
        "log_a",
        "H",
        "Hdot",
        "rho_dot",
        "Hddot",
        "quadrature_error",
    ]
]

,success,rho,t,log_a,H,Hdot,rho_dot,Hddot,quadrature_error
0,True,1.000000e-02,3.841370e+03,3.332232e+04,1.154715e+01,1.000000e-03,-1.732116e-06,-1.732116e-07,4.264777e-11
1,True,1.000000e-04,3.848994e+06,3.333333e+08,1.154701e+02,1.000000e-05,-1.732051e-11,-1.732051e-12,2.289360e-04
2,True,1.000000e-06,3.849002e+09,3.333333e+12,1.154701e+03,1.000000e-07,-1.732051e-16,-1.732051e-17,2.611527e-04
3,True,1.000000e-08,3.849002e+12,3.333333e+16,1.154701e+04,1.000000e-09,-1.732051e-21,-1.732051e-22,2.288487e+02
4,True,1.000000e-10,3.849002e+15,3.333333e+20,1.154701e+05,1.000000e-11,-1.732051e-26,-1.732051e-27,4.366559e+01
5,True,1.000000e-12,3.849002e+18,3.333333e+24,1.154701e+06,1.000000e-13,-1.732051e-31,-1.732051e-32,2.623116e+05


In [2]:
from src.barotropic_analysis import (
    compute_critical_densities,
    infer_limit_evidence,
    scan_density_targets,
)

from src.classifiers import (
    classify_barotropic_power_diffusion,
)

from src.models import (
    BarotropicPowerDiffusionModel,
    BarotropicPowerDiffusionParams,
)

In [3]:
params_F2 = BarotropicPowerDiffusionParams(
    omega=-1.2,
    alpha=4.0,
    beta=-1.0,
    rho0=1.0,
    a0=1.0,
    t0=0.0,
)

In [4]:
F2_density_scan = scan_density_targets(
    params=params_F2,
    rho_targets=[
        1e-2,
        1e-4,
        1e-6,
        1e-8,
        1e-10,
        1e-12,
    ],
    require_future=True,
    strict=False,
)

In [5]:
F2_evidence = infer_limit_evidence(
    F2_density_scan
)

F2_evidence

{'time_limit': 'infinite',
 'rho_limit': 'zero',
 'a_limit': 'diverges',
 'H_limit': 'diverges',
 'Hdot_limit': 'zero',
 'rho_dot_limit': 'zero',
 'Hddot_limit': 'zero',
 'stable_under_threshold_changes': True,
 'confidence': 0.95,
 'target_direction': 'decay_to_zero',
 'estimated_limit_time': inf,
 'diagnostics': {'time': {'time_limit': 'infinite',
   'estimated_limit_time': inf,
   'time_increment_ratio': 1000.0,
   'time_convergence_stable': True},
  'a': {'limit': 'diverges',
   'step_ratio': 10000.000000000002,
   'total_log_change': 3.333333333333334e+24,
   'last_log_step': 3.3330000000000007e+24},
  'H': {'limit': 'diverges',
   'ratio': 99998.75002343701,
   'log_slope': 0.9999999999999839,
   'tail_relative_spread': 0.9989999999999999,
   'increasing_fraction': 1.0,
   'decreasing_fraction': 0.0},
  'Hdot': {'limit': 'zero',
   'ratio': 1e-10,
   'log_slope': -1.9999999999999996,
   'tail_relative_spread': 0.999999,
   'increasing_fraction': 0.0,
   'decreasing_fraction': 1.0

In [6]:
model_F2 = BarotropicPowerDiffusionModel(
    params_F2
)

background_F2 = model_F2.solve_background_t(
    t_end=20.0,
    n_points=2000,
    rho_min=1e-12,
    rho_max=1e12,
    a_max=1e100,
    boundary_tol=1e-8,
    max_step=0.02,
)

In [7]:
classification_F2 = (
    classify_barotropic_power_diffusion(
        background=background_F2,
        params=params_F2,
        limit_evidence=F2_evidence,
    )
)

classification_F2.as_dict()

{'standard_type': None,
 'behavior_label': 'new_behavior_candidate',
 'status': 'new_behavior_candidate',
 'is_singularity': None,
 'finite_time': None,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'diverges',
  'rho': 'zero',
  'a': 'diverges',
  'H': 'diverges',
  'Hdot': 'zero',
  'rho_dot': 'zero',
  'Hddot': 'zero',
  'termination_event': None,
  'solver_success': True,
  'reached_t_end': True,
  'H_zero_crossings': 0,
  'expansion_to_contraction': False,
  'initial_H_sign': 1,
  'final_H_sign': 1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 1.2909944487358056,
   'Hdot': 0.09999999999999998},
  'final_values': {'t': 20.0,
   'rho': 0.281486747565625,
   'a': 8037918025434863.0,
   'H': 2.1978585950352323,
   'Hdot': 0.028148674756562492},
  'tail_relative_spread': {'rho': 0.05388243440040045,
   'a': 0.9866883165243765,
   'H': 0.026208241527267494,
   'Hdot': 0.05388243440040046}},
 'evidence': {'time_limit': 'infinite',
  'rho_limit': 'zero',

In [8]:
F2_automatic_checks = {
    "all_targets_successful": bool(
        F2_density_scan[
            "success"
        ].all()
    ),
    "time_limit_determined": (
        F2_evidence[
            "time_limit"
        ]
        in {
            "finite",
            "infinite",
        }
    ),
    "rho_limit_determined": (
        F2_evidence[
            "rho_limit"
        ]
        == "zero"
    ),
    "classification_created": (
        classification_F2
        is not None
    ),
}

F2_automatic_checks

{'all_targets_successful': True,
 'time_limit_determined': True,
 'rho_limit_determined': True,
 'classification_created': True}

In [19]:
from src.barotropic_analysis import (
    compute_critical_densities,
    identify_boundary_type,
    infer_boundary_evidence,
    scan_boundary_tolerances,
    infer_signed_collapse_evidence,
)

from src.models import (
    BarotropicPowerDiffusionParams,
)

In [11]:
params_D_boundary = BarotropicPowerDiffusionParams(
    omega=0.0,
    alpha=0.25,
    beta=-1.0,
    rho0=1.0,
    a0=1.0,
    t0=0.0,
)

In [12]:
compute_critical_densities(
    params_D_boundary
).as_dict()

{'rho_F': None,
 'rho_D': 0.5,
 'F_degenerate': False,
 'D_degenerate': False,
 'notes': ''}

In [13]:
D_boundary_scan = (
    scan_boundary_tolerances(
        params=params_D_boundary,
        t_end=20.0,
        boundary_tolerances=[
            1e-2,
            1e-3,
            1e-4,
            1e-5,
            1e-6,
            1e-7,
        ],
        n_points=2000,
        rho_min=1e-10,
        rho_max=1e10,
        a_max=1e50,
        max_step=0.01,
    )
)

In [14]:
D_boundary_scan[
    [
        "boundary_tol",
        "success",
        "termination_event",
        "t_final",
        "rho_final",
        "a_final",
        "H_final",
        "Hdot_final",
        "D_final",
        "rho_dot_final",
        "Hddot_final",
    ]
]

,boundary_tol,success,termination_event,t_final,rho_final,a_final,H_final,Hdot_final,D_final,rho_dot_final,Hddot_final
0,1.000000e-02,True,rho_dot_denominator_boundary,0.174354,0.502519,1.111867,0.577354,-0.251259,1.000000e-02,-8.703938e+01,4.351969e+01
1,1.000000e-03,True,rho_dot_denominator_boundary,0.174368,0.500250,1.111876,0.577350,-0.250125,1.000000e-03,-8.664588e+02,4.332294e+02
2,1.000000e-04,True,rho_dot_denominator_boundary,0.174368,0.500025,1.111876,0.577350,-0.250013,1.000000e-04,-8.660687e+03,4.330344e+03
3,1.000000e-05,True,rho_dot_denominator_boundary,0.174368,0.500003,1.111876,0.577350,-0.250001,9.999989e-06,-8.660307e+04,4.330153e+04
4,1.000000e-06,True,rho_dot_denominator_boundary,0.174368,0.500000,1.111876,0.577350,-0.250000,1.000087e-06,-8.659505e+05,4.329752e+05
5,1.000000e-07,True,rho_dot_denominator_boundary,0.174368,0.500000,1.111876,0.577350,-0.250000,1.046678e-07,-8.274040e+06,4.137020e+06


In [15]:
D_boundary_evidence = (
    infer_boundary_evidence(
        D_boundary_scan
    )
)

D_boundary_evidence

{'time_limit': 'finite',
 'rho_limit': 'finite',
 'a_limit': 'finite',
 'H_limit': 'finite',
 'Hdot_limit': 'finite',
 'rho_dot_limit': 'diverges',
 'Hddot_limit': 'diverges',
 'stable_under_tolerance_changes': True,
 'confidence': 0.95,
 'boundary_type': 'D',
 'estimated_limit_time': 0.17436822637378677,
 'estimated_boundary_density': 0.5000009254016473,
 'estimated_log_a': 0.10604906018280376,
 'diagnostics': {'time': {'converges': True,
   'limit': 0.17436822637378677,
   'relative_spread': 1.4432066652858566e-11},
  'rho': {'converges': True,
   'limit': 0.5000009254016473,
   'relative_spread': 2.4738490990161566e-06},
  'log_a': {'converges': True,
   'limit': 0.10604906018280376,
   'relative_spread': 8.332390333265494e-12},
  'H_finite': {'converges': True,
   'limit': 0.5773502691908408,
   'relative_spread': 3.6080027854268337e-12},
  'H_zero': {'tends_to_zero': False,
   'log_slope': -3.9401136447789005e-07,
   'intercept': -0.23855839278064941,
   'decay_ratio': 0.999993686

In [4]:
from src.classifiers import (
    classify_barotropic_power_diffusion,
)

from src.models import (
    BarotropicPowerDiffusionModel,
)

In [17]:
model_D_boundary = (
    BarotropicPowerDiffusionModel(
        params_D_boundary
    )
)

background_D_boundary = (
    model_D_boundary.solve_background_t(
        t_end=20.0,
        n_points=2000,
        rho_min=1e-10,
        rho_max=1e10,
        a_max=1e50,
        boundary_tol=1e-7,
        max_step=0.01,
    )
)

In [19]:
classification_D_boundary = (
    classify_barotropic_power_diffusion(
        background=(
            background_D_boundary
        ),
        params=params_D_boundary,
        limit_evidence=(
            D_boundary_evidence
        ),
    )
)

classification_D_boundary.as_dict()

{'standard_type': 'Type IV',
 'behavior_label': 'higher_derivative_boundary',
 'status': 'robust',
 'is_singularity': True,
 'finite_time': True,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'finite',
  'rho': 'finite',
  'a': 'finite',
  'H': 'finite',
  'Hdot': 'finite',
  'rho_dot': 'diverges',
  'Hddot': 'diverges',
  'termination_event': 'rho_dot_denominator_boundary',
  'solver_success': True,
  'reached_t_end': False,
  'H_zero_crossings': 0,
  'expansion_to_contraction': False,
  'initial_H_sign': 1,
  'final_H_sign': 1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 0.6454972243679028,
   'Hdot': -0.5},
  'final_values': {'t': 0.1743682263675235,
   'rho': 0.500000022901606,
   'a': 1.1118764240251582,
   'H': 0.5773502691896261,
   'Hdot': -0.250000011450803},
  'tail_relative_spread': {'rho': 0.1684187546147451,
   'a': 0.010012858719305563,
   'H': 0.008419848900824731,
   'Hdot': 0.1684187546147451}},
 'evidence': {'time_limit': 'finite',
 

In [16]:
from src.barotropic_analysis import (
    compute_critical_densities,
    initial_density_direction,
    solve_signed_background_compatible,
    summarize_signed_background,
    scan_signed_collapse_thresholds,
)

In [21]:
compute_critical_densities(
    params_D_boundary
).as_dict()

{'rho_F': None,
 'rho_D': 0.5,
 'F_degenerate': False,
 'D_degenerate': False,
 'notes': ''}

In [22]:
initial_density_direction(
    params_D_boundary
)

-1

In [23]:
D_boundary_scan = scan_boundary_tolerances(
    params=params_D_boundary,
    t_end=20.0,
    boundary_tolerances=[
        1e-2,
        1e-3,
        1e-4,
        1e-5,
        1e-6,
        1e-7,
    ],
    n_points=2000,
    rho_min=1e-10,
    rho_max=1e10,
    a_max=1e50,
    max_step=0.001,
)

In [24]:
D_boundary_scan[
    [
        "boundary_tol",
        "success",
        "termination_event",
        "t_final",
        "rho_final",
        "a_final",
        "H_final",
        "Hdot_final",
        "D_final",
        "rho_dot_final",
        "Hddot_final",
    ]
]

,boundary_tol,success,termination_event,t_final,rho_final,a_final,H_final,Hdot_final,D_final,rho_dot_final,Hddot_final
0,1.000000e-02,True,rho_dot_denominator_boundary,0.174354,0.502519,1.111867,0.577354,-0.251259,1.000000e-02,-8.703938e+01,4.351969e+01
1,1.000000e-03,True,rho_dot_denominator_boundary,0.174368,0.500250,1.111876,0.577350,-0.250125,1.000000e-03,-8.664588e+02,4.332294e+02
2,1.000000e-04,True,rho_dot_denominator_boundary,0.174368,0.500025,1.111876,0.577350,-0.250013,1.000000e-04,-8.660687e+03,4.330344e+03
3,1.000000e-05,True,rho_dot_denominator_boundary,0.174368,0.500003,1.111876,0.577350,-0.250001,1.000001e-05,-8.660291e+04,4.330146e+04
4,1.000000e-06,True,rho_dot_denominator_boundary,0.174368,0.500000,1.111876,0.577350,-0.250000,9.992819e-07,-8.666482e+05,4.333241e+05
5,1.000000e-07,True,rho_dot_denominator_boundary,0.174368,0.500000,1.111876,0.577350,-0.250000,9.468372e-08,-9.146508e+06,4.573254e+06


In [25]:
D_boundary_evidence = infer_boundary_evidence(
    D_boundary_scan
)

D_boundary_evidence

{'time_limit': 'finite',
 'rho_limit': 'finite',
 'a_limit': 'finite',
 'H_limit': 'finite',
 'Hdot_limit': 'finite',
 'rho_dot_limit': 'diverges',
 'Hddot_limit': 'diverges',
 'stable_under_tolerance_changes': True,
 'confidence': 0.95,
 'boundary_type': 'D',
 'estimated_limit_time': 0.1743682263751479,
 'estimated_boundary_density': 0.500000924504025,
 'estimated_log_a': 0.106049060183573,
 'diagnostics': {'time': {'converges': True,
   'limit': 0.1743682263751479,
   'relative_spread': 1.443242747534157e-11},
  'rho': {'converges': True,
   'limit': 0.500000924504025,
   'relative_spread': 2.476349550084578e-06},
  'log_a': {'converges': True,
   'limit': 0.106049060183573,
   'relative_spread': 8.332584622294803e-12},
  'H_finite': {'converges': True,
   'limit': 0.5773502691908408,
   'relative_spread': 3.6082248300317588e-12},
  'H_zero': {'tends_to_zero': False,
   'log_slope': -3.940113645061404e-07,
   'intercept': -0.23855839278064928,
   'decay_ratio': 0.9999936869683246,
  

In [26]:
model_D_boundary = BarotropicPowerDiffusionModel(
    params_D_boundary
)

background_D_boundary = (
    model_D_boundary.solve_background_t(
        t_end=20.0,
        n_points=3000,
        rho_min=1e-10,
        rho_max=1e10,
        a_max=1e50,
        boundary_tol=1e-7,
        max_step=0.001,
    )
)

In [27]:
print(
    background_D_boundary.attrs[
        "termination_event"
    ]
)

rho_dot_denominator_boundary


In [28]:
classification_D_boundary = (
    classify_barotropic_power_diffusion(
        background=background_D_boundary,
        params=params_D_boundary,
        limit_evidence=(
            D_boundary_evidence
        ),
    )
)

classification_D_boundary.as_dict()

{'standard_type': 'Type IV',
 'behavior_label': 'higher_derivative_boundary',
 'status': 'robust',
 'is_singularity': True,
 'finite_time': True,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'finite',
  'rho': 'finite',
  'a': 'finite',
  'H': 'finite',
  'Hdot': 'finite',
  'rho_dot': 'diverges',
  'Hddot': 'diverges',
  'termination_event': 'rho_dot_denominator_boundary',
  'solver_success': True,
  'reached_t_end': False,
  'H_zero_crossings': 0,
  'expansion_to_contraction': False,
  'initial_H_sign': 1,
  'final_H_sign': 1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 0.6454972243679028,
   'Hdot': -0.5},
  'final_values': {'t': 0.1743682263777739,
   'rho': 0.5000000230705431,
   'a': 1.111876424031625,
   'H': 0.5773502691896261,
   'Hdot': -0.25000001153527157},
  'tail_relative_spread': {'rho': 0.1685419704739289,
   'a': 0.010027947750147625,
   'H': 0.008433251784310251,
   'Hdot': 0.1685419704739289}},
 'evidence': {'time_limit': 'finite',

In [6]:
params_turnaround = (
    BarotropicPowerDiffusionParams(
        omega=0.0,
        alpha=-0.5,
        beta=0.5,
        rho0=1.0,
        a0=1.0,
        t0=0.0,
    )
)

In [7]:
compute_critical_densities(
    params_turnaround
).as_dict()

{'rho_F': 0.25,
 'rho_D': 0.0625,
 'F_degenerate': False,
 'D_degenerate': False,
 'notes': ''}

In [10]:
signed_turnaround = (
    solve_signed_background_compatible(
        params_turnaround,
        t_end=20.0,
        n_points=5000,
        rho_min=1e-12,
        rho_max=1e7,
        a_min=1e-12,
        a_max=1e100,
        H_max=1e100,
        boundary_tol=1e-8,
        max_step=0.001,
    )
)

In [ ]:
turnaround_summary = (
    summarize_signed_background(
        signed_turnaround
    )
)



{'turnaround_detected': True,
 'turnaround_time': 1.905158688840598,
 'turnaround_density': 0.2500000087985915,
 'turnaround_scale_factor': 1.343705972681404,
 'maximum_a': 1.3437059796330164,
 'time_at_maximum_a': 1.9052639326662422,
 'initial_H': 0.408248290463863,
 'final_H': -1825.5975150206993,
 'final_rho': 9999999.999986755,
 'final_a': 0.005483100308455998,
 'final_Hdot': -4999999.999993377,
 'contracting_at_end': True,
 'termination_event': 'rho_max',
 'solver_success': True,
 'reached_t_end': False,
 'maximum_constraint_residual': 1.3417350381536641e-06}

In [15]:
import numpy as np

signed_turnaround[
    [
        "t",
        "rho",
        "a",
        "H",
        "Hdot",
    ]
].iloc[
    np.linspace(
        0,
        len(signed_turnaround) - 1,
        10,
        dtype=int,
    )
]

,t,rho,a,H,Hdot
0,0.000000,1.000000e+00,1.000000,0.408248,-5.000000e-01
555,0.553334,4.899177e-01,1.181034,0.215984,-2.449588e-01
1110,1.106668,3.164394e-01,1.289078,0.108281,-1.582197e-01
1666,1.660999,2.556414e-01,1.338690,0.030749,-1.278207e-01
2221,2.214333,2.590976e-01,1.335654,-0.039113,-1.295488e-01
2777,2.768665,3.291997e-01,1.279601,-0.118772,-1.645999e-01
3332,3.321999,5.240662e-01,1.163950,-0.232454,-2.620331e-01
3888,3.876330,1.117736e+00,0.972311,-0.443141,-5.588681e-01
4443,4.429664,4.358687e+00,0.667719,-1.051160,-2.179344e+00
4999,4.983995,1.000000e+07,0.005483,-1825.597515,-5.000000e+06


In [17]:
turnaround_collapse_scan = (
    scan_signed_collapse_thresholds(
        params=params_turnaround,
        t_end=20.0,
        rho_max_values=[
            1e2,
            1e3,
            1e4,
            1e5,
            1e6,
            1e7,
        ],
        n_points=5000,
        rho_min=1e-12,
        a_min=1e-12,
        a_max=1e100,
        H_max=1e100,
        boundary_tol=1e-8,
        max_step=0.001,
    )
)

In [18]:
turnaround_collapse_scan[
    [
        "rho_max",
        "success",
        "turnaround_detected",
        "termination_event",
        "t_final",
        "rho_final",
        "a_final",
        "H_final",
        "Hdot_final",
        "maximum_constraint_residual",
    ]
]

,rho_max,success,turnaround_detected,termination_event,t_final,rho_final,a_final,H_final,Hdot_final,maximum_constraint_residual
0,100.0,True,True,rho_max,4.868878,1.000000e+02,0.250310,-5.627314,-5.000000e+01,2.818113e-06
1,1000.0,True,True,rho_max,4.947845,1.000000e+03,0.117515,-18.112506,-5.000000e+02,7.367089e-07
2,10000.0,True,True,rho_max,4.972813,1.000000e+04,0.054743,-57.590508,-5.000000e+03,4.096486e-07
3,100000.0,True,True,rho_max,4.980709,1.000000e+05,0.025438,-182.429791,-5.000000e+04,6.184762e-07
4,1000000.0,True,True,rho_max,4.983205,1.000000e+06,0.011812,-577.205914,-5.000000e+05,3.807105e-07
5,10000000.0,True,True,rho_max,4.983995,1.000000e+07,0.005483,-1825.597515,-5.000000e+06,1.341735e-06


In [20]:
turnaround_evidence = (
    infer_signed_collapse_evidence(
        turnaround_collapse_scan
    )
)

turnaround_evidence

{'time_limit': 'finite',
 'rho_limit': 'diverges',
 'a_limit': 'zero',
 'H_limit': 'diverges',
 'Hdot_limit': 'diverges',
 'rho_dot_limit': 'diverges',
 'Hddot_limit': 'diverges',
 'stable_under_threshold_changes': True,
 'confidence': 0.95,
 'turnaround_detected': True,
 'contracting_branch': True,
 'estimated_limit_time': 4.984360123802502,
 'diagnostics': {'time': {'time_limit': 'finite',
   'estimated_limit_time': 4.984360123802502,
   'time_increment_ratio': 0.31622734442928885,
   'time_convergence_stable': True},
  'rho': {'diverges': True,
   'log_slope': 0.9999999999999077,
   'intercept': 3.1219629378521506e-13,
   'growth_factor': 99999.99999986768,
   'increasing_fraction': 1.0},
  'a_zero': {'tends_to_zero': True,
   'log_slope': -0.3320984378339714,
   'intercept': 0.06506285666501045,
   'decay_ratio': 0.021905273772568765,
   'decreasing_fraction': 1.0},
  'H': {'diverges': True,
   'log_slope': 0.501894881306211,
   'intercept': -0.24978326631126224,
   'growth_factor'

In [21]:
model_turnaround = (
    BarotropicPowerDiffusionModel(
        params_turnaround
    )
)

expanding_turnaround = (
    model_turnaround.solve_background_t(
        t_end=20.0,
        n_points=2500,
        rho_min=1e-12,
        rho_max=1e8,
        a_max=1e100,
        boundary_tol=1e-8,
        max_step=0.001,
    )
)

In [22]:
expanding_turnaround.attrs[
    "termination_event"
]

'effective_density_boundary'

In [23]:
classification_turnaround = (
    classify_barotropic_power_diffusion(
        background=expanding_turnaround,
        params=params_turnaround,
        signed_background=(
            signed_turnaround
        ),
        limit_evidence=(
            turnaround_evidence
        ),
    )
)

classification_turnaround.as_dict()

{'standard_type': None,
 'behavior_label': 'turnaround_to_big_crunch',
 'status': 'robust',
 'is_singularity': True,
 'finite_time': True,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'finite',
  'rho': 'diverges',
  'a': 'zero',
  'H': 'diverges',
  'Hdot': 'diverges',
  'rho_dot': 'diverges',
  'Hddot': 'diverges',
  'termination_event': 'rho_max',
  'solver_success': True,
  'reached_t_end': False,
  'H_zero_crossings': 1,
  'expansion_to_contraction': True,
  'initial_H_sign': 1,
  'final_H_sign': -1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 0.408248290463863,
   'Hdot': -0.5},
  'final_values': {'t': 4.983994976137386,
   'rho': 9999999.999986755,
   'a': 0.005483100308455998,
   'H': -1825.5975150206993,
   'Hdot': -4999999.999993377},
  'tail_relative_spread': {'rho': 0.9999994596116738,
   'a': 0.9912498331459905,
   'H': 0.9993486750950602,
   'Hdot': 0.9999994596116738},
  'maximum_constraint_residual': 1.3417350381536641e-06},
 'evidenc